# 🏠 Simulador de Financiamento Imobiliário — California Housing

**Notebook:** 03 de 03  
**Objetivo:** Ferramenta interativa para simular a viabilidade de compra ou aluguel em diferentes zonas da Califórnia (dados de 1990), com base na sua renda, entrada disponível e prazo de financiamento.

---

## Como usar
1. Execute todas as células (`Run All`)
2. Use os **sliders e dropdowns** para ajustar seu perfil
3. O painel atualiza instantaneamente com prestação, comprometimento de renda, zona recomendada e comparativo compra × aluguel

---
> ⚠️ Valores baseados no California Housing Dataset de 1990. Use como referência histórica/educacional.

In [ ]:
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ── Dados ──────────────────────────────────────────────────────────────────
df = pd.read_csv('../housing.csv')
df.columns = ['lon','lat','idade','total_q','total_d','pop','dom',
              'renda','valor','zona']
df['total_d'] = df['total_d'].fillna(df['total_d'].median())
df['renda_ano'] = df['renda'] * 10_000
df['aluguel_est'] = df['valor'] * 0.005  # regra dos 0.5% ao mês

# Estatísticas por zona
zona_stats = df.groupby('zona').agg(
    valor_med=('valor','median'),
    valor_p25=('valor', lambda x: x.quantile(0.25)),
    valor_p75=('valor', lambda x: x.quantile(0.75)),
    renda_med=('renda_ano','median'),
    aluguel_med=('aluguel_est','median'),
    n=('valor','count'),
).sort_values('valor_med')

# ── Constantes de design ───────────────────────────────────────────────────
ZONE_COLOR = {
    'INLAND':     '#F5A623',
    '<1H OCEAN':  '#34D399',
    'NEAR OCEAN': '#0EA5E9',
    'NEAR BAY':   '#A78BFA',
    'ISLAND':     '#F43F5E',
}
BG   = '#0E1117'
GOLD = '#F5A623'
GRN  = '#34D399'
RED  = '#F43F5E'
BLU  = '#0EA5E9'

print("✅ Dados carregados:", len(df), "distritos")
print(zona_stats[['valor_med','renda_med','aluguel_med','n']].to_string())

In [ ]:
def calcular_prestacao(valor_imovel, entrada_pct, taxa_anual, prazo_anos):
    """Price/HP tabela SAC → retorna prestação mensal inicial."""
    financiado = valor_imovel * (1 - entrada_pct / 100)
    taxa_mes = taxa_anual / 100 / 12
    n = prazo_anos * 12
    if taxa_mes == 0:
        return financiado / n, financiado
    prestacao = financiado * taxa_mes * (1 + taxa_mes)**n / ((1 + taxa_mes)**n - 1)
    return prestacao, financiado

def custo_total_compra(valor_imovel, entrada_pct, taxa_anual, prazo_anos):
    prestacao, financiado = calcular_prestacao(valor_imovel, entrada_pct, taxa_anual, prazo_anos)
    entrada_valor = valor_imovel * entrada_pct / 100
    total_pago = entrada_valor + prestacao * prazo_anos * 12
    juros_total = total_pago - valor_imovel
    return prestacao, entrada_valor, total_pago, juros_total

def zonas_viaveis(renda_anual, entrada_pct, taxa_anual, prazo_anos, max_comprometimento=0.30):
    """Retorna quais zonas são acessíveis dado o perfil."""
    renda_mensal = renda_anual / 12
    limite_prestacao = renda_mensal * max_comprometimento
    result = []
    for zona, row in zona_stats.iterrows():
        for percentil, label in [(row['valor_p25'],'P25'),(row['valor_med'],'Mediana'),(row['valor_p75'],'P75')]:
            prestacao, _ = calcular_prestacao(percentil, entrada_pct, taxa_anual, prazo_anos)
            comprometimento = prestacao / renda_mensal
            result.append({
                'zona': zona,
                'percentil': label,
                'valor': percentil,
                'prestacao': prestacao,
                'comprometimento': comprometimento,
                'viavel': comprometimento <= max_comprometimento,
            })
    return pd.DataFrame(result)

print("✅ Funções financeiras carregadas")

In [ ]:
def render_dashboard(renda_anual, entrada_pct, taxa_anual, prazo_anos,
                     zona_escolhida, max_comp):
    plt.style.use('dark_background')
    plt.rcParams.update({'figure.facecolor': BG, 'axes.facecolor': BG,
                         'axes.spines.top': False, 'axes.spines.right': False,
                         'grid.color': '#1E283C', 'grid.linewidth': 0.6,
                         'font.size': 11})

    valor_zona   = zona_stats.loc[zona_escolhida, 'valor_med']
    aluguel_zona = zona_stats.loc[zona_escolhida, 'aluguel_med']
    renda_mensal = renda_anual / 12

    prestacao, entrada_val, total_pago, juros_total = custo_total_compra(
        valor_zona, entrada_pct, taxa_anual, prazo_anos)

    comprometimento = prestacao / renda_mensal
    comp_aluguel    = aluguel_zona / renda_mensal
    viavel_compra   = comprometimento <= max_comp / 100
    viavel_aluguel  = comp_aluguel    <= max_comp / 100

    fig = plt.figure(figsize=(16, 10), facecolor=BG)
    gs  = fig.add_gridspec(3, 3, hspace=0.45, wspace=0.35)

    # ── KPI boxes (linha 1) ────────────────────────────────────────────────
    kpis = [
        ('Prestação Mensal', f'${prestacao:,.0f}', GRN if viavel_compra else RED),
        ('Comprometimento', f'{comprometimento*100:.1f}%', GRN if viavel_compra else RED),
        ('Aluguel Estimado', f'${aluguel_zona:,.0f}/mês', GRN if viavel_aluguel else RED),
        ('Entrada Necessária', f'${entrada_val:,.0f}', BLU),
        ('Total Financiado', f'${total_pago:,.0f}', GOLD),
        ('Juros Pagos', f'${juros_total:,.0f}', '#A78BFA'),
    ]
    for i, (label, value, color) in enumerate(kpis):
        ax_kpi = fig.add_subplot(gs[0, i % 3] if i < 3 else gs[1, i % 3])
        ax_kpi.set_facecolor('#131720')
        ax_kpi.set_xlim(0, 1); ax_kpi.set_ylim(0, 1)
        ax_kpi.axis('off')
        ax_kpi.text(0.5, 0.65, value, ha='center', va='center',
                    fontsize=22, fontweight='bold', color=color)
        ax_kpi.text(0.5, 0.25, label, ha='center', va='center',
                    fontsize=10, color='#888')
        for spine in ['top','bottom','left','right']:
            ax_kpi.spines[spine].set_visible(False)

    # ── Comparativo zonas (linha 2, col 0-1) ─────────────────────────────
    ax_zonas = fig.add_subplot(gs[2, :2])
    df_viab = zonas_viaveis(renda_anual, entrada_pct, taxa_anual, prazo_anos, max_comp/100)
    df_med  = df_viab[df_viab['percentil'] == 'Mediana'].sort_values('comprometimento')

    colors_bar = [ZONE_COLOR.get(z,'#888') for z in df_med['zona']]
    alpha_bar  = [1.0 if v else 0.35 for v in df_med['viavel']]
    bars = ax_zonas.barh(df_med['zona'], df_med['comprometimento']*100,
                         color=colors_bar, edgecolor='none')
    for bar, alph in zip(bars, alpha_bar):
        bar.set_alpha(alph)
    ax_zonas.axvline(max_comp, color=RED, lw=1.5, linestyle='--',
                     label=f'Limite {max_comp}%')
    ax_zonas.set_xlabel('Comprometimento de Renda (%)')
    ax_zonas.set_title(f'Comprometimento por Zona — Prestação Mediana\n(entrada {entrada_pct}%, {prazo_anos} anos, taxa {taxa_anual}%)')
    ax_zonas.legend(fontsize=9)
    for bar, row in zip(bars, df_med.itertuples()):
        ax_zonas.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                      f'{row.comprometimento*100:.1f}%', va='center', fontsize=9,
                      color=GRN if row.viavel else RED)

    # ── Compra vs Aluguel ao longo do tempo (linha 2, col 2) ────────────
    ax_cv = fig.add_subplot(gs[2, 2])
    meses = np.arange(1, prazo_anos*12+1)
    custo_acum_compra  = entrada_val + prestacao * meses
    custo_acum_aluguel = aluguel_zona * meses

    ax_cv.plot(meses/12, custo_acum_compra/1000,  color=BLU,  lw=2, label='Compra (acumulado)')
    ax_cv.plot(meses/12, custo_acum_aluguel/1000, color=GOLD, lw=2, label='Aluguel (acumulado)')
    ax_cv.fill_between(meses/12, custo_acum_compra/1000, custo_acum_aluguel/1000,
                        alpha=0.1, color=GRN if custo_acum_compra[-1] < custo_acum_aluguel[-1] else RED)

    crossover = np.where(custo_acum_compra <= custo_acum_aluguel)[0]
    if len(crossover) > 0:
        cx = crossover[0] / 12
        ax_cv.axvline(cx, color=GRN, lw=1, linestyle=':',
                      label=f'Break-even: {cx:.1f} anos')

    ax_cv.set_title('Compra vs Aluguel — Custo Acumulado')
    ax_cv.set_xlabel('Anos'); ax_cv.set_ylabel('Custo Acumulado (k$)')
    ax_cv.legend(fontsize=8)

    zona_col = ZONE_COLOR.get(zona_escolhida, '#888')
    rec = '✅ VIÁVEL' if viavel_compra else '❌ INVIÁVEL'
    fig.suptitle(f'Zona Selecionada: {zona_escolhida}  |  Compra: {rec}  |  '
                 f'Renda anual: ${renda_anual:,.0f}',
                 fontsize=13, color=zona_col, y=1.01)

    plt.tight_layout()
    plt.show()

print("✅ Função de dashboard carregada")

In [ ]:
# ── Widgets ────────────────────────────────────────────────────────────────
style  = {'description_width': '180px'}
layout = widgets.Layout(width='500px')

w_renda = widgets.IntSlider(
    value=60000, min=20000, max=300000, step=5000,
    description='Renda Anual ($)',
    style=style, layout=layout,
    readout_format=',d',
)
w_entrada = widgets.IntSlider(
    value=20, min=5, max=50, step=5,
    description='Entrada (%)',
    style=style, layout=layout,
)
w_taxa = widgets.FloatSlider(
    value=8.0, min=3.0, max=15.0, step=0.5,
    description='Taxa de Juros Anual (%)',
    style=style, layout=layout,
)
w_prazo = widgets.IntSlider(
    value=30, min=5, max=40, step=5,
    description='Prazo (anos)',
    style=style, layout=layout,
)
w_zona = widgets.Dropdown(
    options=list(zona_stats.index),
    value='INLAND',
    description='Zona Californiana',
    style=style, layout=widgets.Layout(width='400px'),
)
w_maxcomp = widgets.IntSlider(
    value=30, min=20, max=50, step=5,
    description='Comprometimento Máx (%)',
    style=style, layout=layout,
)

ui = widgets.VBox([
    widgets.HTML('<h3 style="color:#F5A623;font-family:monospace">⚙️ Configurações do Simulador</h3>'),
    widgets.HBox([
        widgets.VBox([w_renda, w_entrada, w_taxa]),
        widgets.VBox([w_prazo, w_zona, w_maxcomp]),
    ]),
    widgets.HTML('<hr style="border-color:#333">'),
])

out = widgets.Output()

def on_change(_):
    with out:
        clear_output(wait=True)
        render_dashboard(
            renda_anual=w_renda.value,
            entrada_pct=w_entrada.value,
            taxa_anual=w_taxa.value,
            prazo_anos=w_prazo.value,
            zona_escolhida=w_zona.value,
            max_comp=w_maxcomp.value,
        )

for w in [w_renda, w_entrada, w_taxa, w_prazo, w_zona, w_maxcomp]:
    w.observe(on_change, names='value')

display(ui, out)
on_change(None)  # render inicial

## Análise de Sensibilidade — Prestação × Taxa × Prazo

Heatmaps mostrando como a prestação mensal varia conforme taxa e prazo, para cada zona.

In [ ]:
import seaborn as sns

taxas  = np.arange(4.0, 14.5, 1.0)
prazos = np.arange(10, 41, 5)

zonas_plot = [z for z in ['INLAND','<1H OCEAN','NEAR OCEAN','NEAR BAY'] if z in zona_stats.index]
fig, axes = plt.subplots(1, len(zonas_plot), figsize=(5*len(zonas_plot), 5))

for ax, zona in zip(axes, zonas_plot):
    valor = zona_stats.loc[zona, 'valor_med']
    grid = np.zeros((len(taxas), len(prazos)))
    for i, t in enumerate(taxas):
        for j, p in enumerate(prazos):
            grid[i, j], _ = calcular_prestacao(valor, 20, t, p)

    sns.heatmap(grid/1000, annot=True, fmt='.1f', ax=ax,
                xticklabels=prazos, yticklabels=[f'{t:.0f}%' for t in taxas],
                cmap='RdYlGn_r', linewidths=0.3, linecolor='#1E283C',
                cbar_kws={'label':'Prestação (k$/mês'})
    ax.set_title(f'{zona}\n(valor med: ${valor/1e3:.0f}k, entrada 20%)',
                 fontsize=10, color=ZONE_COLOR.get(zona,'white'))
    ax.set_xlabel('Prazo (anos)')
    ax.set_ylabel('Taxa Anual' if ax is axes[0] else '')

plt.suptitle('Sensibilidade: Prestação Mensal (k$) por Taxa × Prazo\n(entrada 20%)',
             fontsize=13, y=1.02)
plt.tight_layout()

FIGURES = Path('../figures')
FIGURES.mkdir(exist_ok=True)
plt.savefig(FIGURES / 'sim01_sensibilidade_prestacao.png', bbox_inches='tight',
            dpi=130, facecolor=BG)
plt.show()

In [ ]:
# Renda mínima necessária para financiar em cada zona (entrada 20%, prazo 30 anos)
rendas = np.linspace(20000, 300000, 500)
taxa_ref, prazo_ref, entrada_ref = 8.0, 30, 20

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Gráfico 1: Curva de acessibilidade — renda × % de comprometimento por zona
for zona in zonas_plot:
    valor = zona_stats.loc[zona, 'valor_med']
    prestacao, _ = calcular_prestacao(valor, entrada_ref, taxa_ref, prazo_ref)
    comps = prestacao / (rendas / 12) * 100
    axes[0].plot(rendas/1000, comps, color=ZONE_COLOR.get(zona,'#888'), lw=2, label=zona)

axes[0].axhline(30, color=RED, lw=1.5, linestyle='--', label='Limite 30%')
axes[0].axhline(40, color='#F97316', lw=1, linestyle=':', label='Limite 40%')
axes[0].set_ylim(0, 100)
axes[0].set_title(f'Comprometimento × Renda por Zona\n(entrada {entrada_ref}%, {prazo_ref} anos, taxa {taxa_ref}%)')
axes[0].set_xlabel('Renda Anual (k$)')
axes[0].set_ylabel('Comprometimento de Renda (%)')
axes[0].legend(fontsize=9)
axes[0].fill_between([20, 300], 30, 100, alpha=0.05, color=RED)
axes[0].fill_between([20, 300],  0,  30, alpha=0.05, color=GRN)

# Gráfico 2: Renda mínima necessária por zona e percentil
zona_labels, rmin_p25, rmin_med, rmin_p75 = [], [], [], []
for zona in zonas_plot:
    zona_labels.append(zona)
    for pct_col, lst in [('valor_p25', rmin_p25), ('valor_med', rmin_med), ('valor_p75', rmin_p75)]:
        valor = zona_stats.loc[zona, pct_col]
        prestacao, _ = calcular_prestacao(valor, entrada_ref, taxa_ref, prazo_ref)
        renda_min = prestacao * 12 / 0.30
        lst.append(renda_min / 1000)

x = np.arange(len(zona_labels))
w = 0.25
axes[1].bar(x - w, rmin_p25, w, color=[ZONE_COLOR.get(z,'#888') for z in zona_labels],
            alpha=0.5, label='P25 (imóvel barato)')
axes[1].bar(x,     rmin_med, w, color=[ZONE_COLOR.get(z,'#888') for z in zona_labels],
            alpha=0.85, label='Mediana')
axes[1].bar(x + w, rmin_p75, w, color=[ZONE_COLOR.get(z,'#888') for z in zona_labels],
            alpha=0.4, label='P75 (imóvel caro)', edgecolor='white', linewidth=0.5)
axes[1].set_xticks(x)
axes[1].set_xticklabels(zona_labels, fontsize=9)
axes[1].set_title('Renda Mínima Anual para Financiar\n(comprometimento ≤ 30%)')
axes[1].set_ylabel('Renda Mínima (k$/ano)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES / 'sim02_renda_minima.png', bbox_inches='tight', dpi=130, facecolor=BG)
plt.show()